In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [ ]:
%run src/ttp/train_eval_model.py dvc_pipes/ttp/params_ttp.yaml

In [6]:
%debug

> /home/jovyan/work/sec_bert/src/funcs.py(210)<listcomp>()
    208         # import pdb;pdb.set_trace()
    209         pred = (proba>thresh).astype(int)
--> 210         p, r, f1, sup = [it[1] for it in precision_recall_fscore_support(y_true, pred)]
    211 
    212         res_metrics_df = pd.DataFrame({'precision':p, 'recall':r, 'f1':f1, 'sup':sup}, index=[thresh])



ipdb>  y_true


*** NameError: name 'y_true' is not defined


ipdb>  pred


*** NameError: name 'pred' is not defined


...

ipdb>  y_true.sum()


0


ipdb>  


0


ipdb>  q


In [ ]:
res_tr_df = pd.DataFrame({'y_proba':np.array(res_tr['pred']).tolist(), 'y':data.loc[tr_idx, 'target'].values.tolist()})

- dvc exp run -S val_only_proc=false

- dvc exp show --drop .* --keep 'Experiment|кол1|кол2' --rev commit

In [11]:
import click


from ruamel.yaml import YAML

import sys
sys.path.append('.')

import numpy as np
import pandas as pd
import joblib
import json

from nltk import word_tokenize
from itertools import chain
import re 

from sklearn.cluster import KMeans


from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD, NMF
from sklearn.feature_extraction.text import CountVectorizer


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn.pipeline import make_pipeline
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import (log_loss, roc_auc_score, average_precision_score, f1_score, 
                            precision_recall_fscore_support, confusion_matrix)
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier


import seaborn as sns
import matplotlib.pyplot as plt

from src.funcs import set_seed
from src.spec_funcs import train_eval_classic

conf = YAML().load(open('params.yaml'))

set_seed(conf['seed'])

conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))

# так как в функцию передается conf с одноименными ключами
# а такие ключи в разных params.yaml файлах dvc не дает задать приходится обходить    
conf_ttp['seed'] = conf['seed']
conf_ttp['prep_text'] = conf['prep_text']
conf_ttp['train_eval_model'] = conf_ttp['train_eval_model_ttp']
conf_ttp['feat_gen'] = conf_ttp['feat_gen_ttp']
conf_ttp['feat_eng'] = conf_ttp['feat_eng_ttp']

set_seed(conf['seed'])


In [12]:
conf = conf_ttp 
target_col='ttp'
thresh_space_l=np.arange(0.005, 1, 0.005)

In [14]:
from src.funcs import metric_multi
from src.funcs import get_conf_df, get_pred_thresh, set_seed, get_opt_thresh


In [15]:
if target_col=='ttp':
    mlb = joblib.load(conf['prep_text']['ttp_mlb_fn'])
else:
    mlb = joblib.load(conf['prep_text']['mlb_fn'])
data = pd.read_csv(conf['feat_gen']['data_fn'])
# feat_data = pd.read_csv(conf['feat_gen']['feat_fn'])
feat_data = pd.read_csv(conf['feat_eng']['feat_final_fn'])

data['target'] = data['target'].map(lambda x: eval(x))
data[target_col] = data[target_col].map(lambda x: eval(x))

tr_idx = data.query('split=="tr"').index
val_idx = data.query('split=="val"').index
ts_idx = data.query('split=="ts"').index

# определяемся с классификатором атак
if conf['train_eval_model']['attack_clf']:
    
    data['is_attack'] = (data[target_col].str.len()>0).astype(np.int8)
    attack_clf = make_pipeline(RobustScaler(), 
                             LogisticRegression(random_state=conf['seed'], class_weight=conf['train_eval_model']['balanced'], max_iter=10000))
    
    Y_at_train = np.array(data.loc[tr_idx, 'is_attack'].values.tolist())
    
    attack_clf.fit(feat_data.loc[tr_idx].values, Y_at_train)
    
    Y_at_tr_proba = attack_clf.predict_proba(feat_data.loc[tr_idx])[:,1]
    
    Y_at_val_proba = attack_clf.predict_proba(feat_data.loc[val_idx])[:, 1]
    Y_at_val = np.array(data.loc[val_idx, 'is_attack'].values.tolist())


    metric = conf['train_eval_model']['opt_metric']
    res_df, thresh = get_pred_thresh(Y_at_val, Y_at_val_proba, opt_metric=metric)
    feat_data['attack_pred'] = (attack_clf.predict_proba(feat_data)[:,1]>thresh).astype(int)

if conf['train_eval_model']['chain']:
    wrap_class = ClassifierChain
else:
    wrap_class = OneVsRestClassifier 
    
if conf['train_eval_model']['model']=='logreg':
    model = LogisticRegression(random_state=conf['seed'], class_weight=conf['train_eval_model']['balanced'], max_iter=10000)
elif conf['train_eval_model']['model']=='tree':
    model = DecisionTreeClassifier(random_state=conf['seed'], class_weight=conf['train_eval_model']['balanced'])
elif conf['train_eval_model']['model']=='boost':
    model = HistGradientBoostingClassifier(random_state=conf['seed'], class_weight=conf['train_eval_model']['balanced'])
elif conf['train_eval_model']['model']=='knn':
    model = KNeighborsClassifier(class_weight=conf['train_eval_model']['balanced'])    
elif conf['train_eval_model']['model']=='forest':
    model = ExtraTreesClassifier(random_state=conf['seed'], class_weight=conf['train_eval_model']['balanced'])

    
clf_pipe = make_pipeline(RobustScaler(), wrap_class(model, random_state=conf['seed'])) if conf['train_eval_model']['chain'] else make_pipeline(RobustScaler(), wrap_class(model))

Y_train = np.array(data.loc[tr_idx, 'target'].values.tolist())

clf_pipe.fit(feat_data.loc[tr_idx].values, Y_train)

Y_tr_proba = clf_pipe.predict_proba(feat_data.loc[tr_idx])    

tr_roc_auc, _ = metric_multi(Y_train, Y_tr_proba, roc_auc_score)
tr_logloss, _ = metric_multi(Y_train, Y_tr_proba, log_loss, labels=[0,1])
tr_pr_auc, _ = metric_multi(Y_train, Y_tr_proba, average_precision_score)

Y_val_proba = clf_pipe.predict_proba(feat_data.loc[val_idx])
Y_val = np.array(data.loc[val_idx, 'target'].values.tolist())


roc_auc, _ = metric_multi(Y_val, Y_val_proba, roc_auc_score)
logloss, _ = metric_multi(Y_val, Y_val_proba, log_loss, labels=[0,1])
pr_auc, _ = metric_multi(Y_val, Y_val_proba, average_precision_score)



/opt/conda/lib/python3.11/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(


In [22]:
# import pdb;pdb.set_trace()
thresh_l = get_opt_thresh(y_true = Y_val, probas = Y_val_proba, mlb = mlb, opt_metric=conf['train_eval_model']['opt_metric'], thresh_space_l=thresh_space_l,
                      dump_fn = conf['train_eval_model']['opt_metric_fn'])

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/m

IndexError: index 1 is out of bounds for axis 0 with size 1

In [23]:
%debug

> /home/jovyan/work/sec_bert/src/funcs.py(210)<listcomp>()
    208         # import pdb;pdb.set_trace()
    209         pred = (proba>thresh).astype(int)
--> 210         p, r, f1, sup = [it[1] for it in precision_recall_fscore_support(y_true, pred)]
    211 
    212         res_metrics_df = pd.DataFrame({'precision':p, 'recall':r, 'f1':f1, 'sup':sup}, index=[thresh])



ipdb>  u


> /home/jovyan/work/sec_bert/src/funcs.py(210)get_pred_thresh()
    208         # import pdb;pdb.set_trace()
    209         pred = (proba>thresh).astype(int)
--> 210         p, r, f1, sup = [it[1] for it in precision_recall_fscore_support(y_true, pred)]
    211 
    212         res_metrics_df = pd.DataFrame({'precision':p, 'recall':r, 'f1':f1, 'sup':sup}, index=[thresh])



ipdb>  u


> /home/jovyan/work/sec_bert/src/funcs.py(229)get_opt_thresh()
    227     for i in range(num_cls):
    228 
--> 229         res_df, thresh = get_pred_thresh(y_true[:,i], probas[:,i], opt_metric, thresh_space_l)
    230 
    231         res_d[i] = res_df



...

ipdb>  l


    205     if len(thresh_space_l)==0:
    206         thresh_space_l = np.arange(0.05, 0.95, 0.05)
    207     for thresh in thresh_space_l:
    208         # import pdb;pdb.set_trace()
    209         pred = (proba>thresh).astype(int)
--> 210         p, r, f1, sup = [it[1] for it in precision_recall_fscore_support(y_true, pred)]
    211 
    212         res_metrics_df = pd.DataFrame({'precision':p, 'recall':r, 'f1':f1, 'sup':sup}, index=[thresh])
    213         res_metrics_l.append(res_metrics_df)
    214     res_df = pd.concat([it for it in res_metrics_l], axis=0)
    215     thresh = res_df.index[res_df[opt_metric].argmax()].round(3)



ipdb>  y_true.sum()


0


ipdb>  q


In [43]:
i = 128

In [44]:
mlb.classes_[128]

'T1590'

In [45]:
np.array(data.query('split=="val"')['target'].values.tolist()).sum(axis=0)

array([ 10,  49,  42,  10,   9,   5,  22,   5,  55,  15,   5,  45,   4,
       199,   4,   3,  40,  78,   6,   4,  40,  10,  30,   7,  16,  55,
        85,  30,  55, 182,   8,  13,  71,  74,   4,  28,  45,  84,  64,
        18,  49,   5,  20,   8,  16, 109,  65,  20,  41,  29,   7,   7,
        12,   7,   5,  13,   5,   4,  20,   7,  15,  10,   7,   6, 120,
        11,  19,   6,   3,  10,  60,   5,   7,   8,   2,  47,  15,   3,
         7,   6,   5,  10,   6,  11,   7,   9,  24,   3,  12,  32,   4,
         3,   6,  38,  25,  73,  16,   6,  16,  18,   3,  27,  14,   6,
         8,  12,  30,   4,  42,  29,   3,  59,   7,   6,  16,  16,   8,
         6,  46,  44,   1,  21,  11,  11,   6,  10,  22,  11,   0,   3,
         3,   2,   3,   9,  12,   8,   4,   3,  86])